# 第14回 世界の空港を、つながりで読む（解答例・教員用）


---


## 前回からの接続


第13回は、年代×ジャンルの数えた表を読みました。今日は、もの同士のつながりそのものを読みます。世界の空港と、空港を結ぶ航空路線です。


---


## 今日の分析目標


世界の空港のつながりから、要の空港と、まとまり（地域）が見えるかに答える。

まず、路線の一覧（表）を、空港を点、路線を線にしたネットワークにします。次に、空港ごとのつながりの数を数え、要の空港を探します。そして、何回の乗り継ぎで世界のどこへでも行けるかを測ります。さらに、つながりだけで群を作り、地域が見えるかを確かめます。最後に、群を作る物差しを替えたり、ハブを外したりして、ネットワークの形を読みます。

もう1つのねらいは、辞書の値にリストをためて、つながり（隣の一覧）を表せるようになることです。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

OpenFlightsというサイトが、世界の空港と航空路線の一覧を公開しています（Open Database License）。路線の一覧は2014年6月で更新が止まっているので、今日は「2014年の世界」を見ます。1行が「どの空港から、どの空港へ、どの航空会社が飛ぶか」です。


### 場面

世界の空港は、路線でつながっています。どの空港が要か。何回乗り継げば、世界のどこへでも行けるか。路線のつながりだけで、地域のまとまりが見えるか。

点（ノード）と線（エッジ）で、つながりを表したものをネットワークといいます。今日は、空港が点、路線が線です。つながりの形を読む分析を、ネットワーク分析といいます。


### 手元のデータ

- 路線の一覧：67,663行。出発の空港と到着の空港を、3文字の記号（IATA）で書く
- 空港の一覧：7,698行。名前・都市・国・緯度・経度・時間帯

長崎空港の記号はNGS、羽田はHND、関西はKIXです。使うときは、"OpenFlights (openflights.org)"のクレジットを書きます。


### 最後に出す答え

どの空港が要か。世界は何回の乗り継ぎで届くか。路線のつながりだけで、地域のまとまりは見えるかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 表からつながりへ：空港を点、路線を線にすると、世界では3,397空港・19,231本のネットワークになります。
2. 要の空港：えっ、アムステルダムは248路線とつながりますが、空港の真ん中は3路線です。
3. 世界の狭さ：長崎から、ほとんどの空港へ3〜4回で行けます。世界全体の平均は4.1回です。
4. まとまり：緯度・経度を使わず、つながりだけで群を作ると、大陸がほぼ出ます。
5. 切り方を替える：えっ、別の物差し（スペクトルクラスタリング）では、3,072空港が1つの群に残ります。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 表からつながりへ（1節） | 一覧はどうネットワークになる？ | 隣を辞書のリストにため、`networkx`のグラフにする |
| 要の空港（2節） | どこが要か？ | つながっている空港の数（次数）を数え、その分布を見る |
| 世界の狭さ（3節） | 何回で行ける？ | 最短の乗り継ぎを数える |
| まとまり（4節） | 地域が分かれるか？ | つながりだけで群を作り、地図に塗る |
| 切り方を替える（5節） | 物差しを替えると？ | スペクトルクラスタリングで切り、ハブを外してみる |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 線の一覧から、辞書の値にリストをためて隣の一覧を作り、`networkx`のグラフにできる
- 次数を数え、ハブと次数の分布（平均と中央値の離れ）を読める
- 最短経路を数え、ネットワークの狭さを測れる
- つながりだけで群を作り、地図や地域と突き合わせて読める
- 群を作る物差しを替えると見つかる群が変わること、ハブが強みで弱みであることを説明できる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す。今日は、ネットワークの道具`networkx`も使う
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. 一覧は、どうネットワークになる？


ここから、段1「表からつながりへ」（1節）です。問いは、路線の一覧（表）を、どうネットワークにすればよいか、です。


### お手本：路線と空港を読む

- 役割：路線と空港の一覧を読み、出発と到着の列に名前を付ける
- 入力：データの置き場所`DATA_DIR`と、2つのファイル
- 出力：表`routes`（67,663行）と`airports`（7,698行）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
routes = pd.read_csv(f'{DATA_DIR}/openflights_routes.dat',
                     header=None)
airports = pd.read_csv(f'{DATA_DIR}/openflights_airports.dat',
                       header=None)
routes = routes.rename(columns={2: '出発', 4: '到着'})
print(len(routes), len(airports))
routes[['出発', '到着']].head(3)


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `pd.read_csv`の2行：路線と空港の一覧を読む。列の名前の行が無いので`header=None`
- `rename`の行：番号2の列を「出発」、番号4の列を「到着」にする（第5回の辞書）
- `print`と最後の行：行の数と、先頭3本の路線


### お手本：空港の情報

- 役割：空港の列に名前を付け、記号で引ける表にし、時間帯から地域を作る
- 入力：表`airports`
- 出力：表`info`（記号が行の名前）


In [ ]:
airports = airports.rename(columns={2: '都市', 3: '国', 4: 'IATA',
                                    6: '緯度', 7: '経度',
                                    11: '時間帯'})
info = airports.dropna(subset=['IATA']).drop_duplicates('IATA')
info = info.set_index('IATA')
info['地域'] = info['時間帯'].str.split('/').str[0]
info.loc[['NGS', 'AMS'], ['都市', '国', '地域']]


#### 1行ずつ読む

- 1〜3行目：列に名前を付ける
- 4・5行目：記号の無い空港と重なりを外し、記号を行の名前にする
- 6行目：時間帯（`Asia/Tokyo`）を「/」で分け、前の部分を地域にする（第13回の`split`）
- 7行目：長崎とアムステルダムの情報を見る


### 線の一覧から、隣の一覧へ

路線の一覧は、線の一覧（どこからどこへ）です。空港ごとに「隣の空港」の一覧にすると、つながりが読みやすくなります。小さな例で、4空港・5本の線（長崎・関西・羽田・ニューヨーク）から、隣の一覧を作ります。


### お手本：隣を辞書のリストにためる

- 役割：線を1本ずつ読み、両端の空港の隣のリストに、相手を足す
- 入力：線の一覧`edges`
- 出力：辞書`nb`（キーが空港、値が隣の空港のリスト）


In [ ]:
edges = [('NGS', 'KIX'), ('KIX', 'JFK'), ('NGS', 'HND'),
         ('HND', 'KIX'), ('HND', 'JFK')]
nb = {}
for a, b in edges:
    if a not in nb:
        nb[a] = []
    if b not in nb:
        nb[b] = []
    nb[a].append(b)
    nb[b].append(a)
print(nb)


#### 1行ずつ読む

- 1・2行目：線の一覧。`('NGS', 'KIX')`は2つの値の組（タプル）で、1本の線
- 3行目：空の辞書`nb`。キーが空港、値が隣の空港のリストになる
- 4行目：線を1本ずつ取り、組の前を`a`、後ろを`b`に入れる
- 5〜8行目：まだキーが無い空港には、空のリストを用意する（`not in`、第6回の`if`）
- 9・10行目：`a`の隣に`b`を、`b`の隣に`a`を足す。路線は両方向に使える
- 11行目：隣の一覧


#### 中身を見る：2本だけ

線を2本だけ読み、1本ごとに辞書を表示します。名前は`nb2`にして、上の`nb`を壊さないようにします。


In [ ]:
nb2 = {}
for a, b in edges[:2]:
    if a not in nb2:
        nb2[a] = []
    if b not in nb2:
        nb2[b] = []
    nb2[a].append(b)
    nb2[b].append(a)
    print(nb2)


1本ごとに、辞書のキーが増え、リストが伸びます。


> 今日の重要概念（Python）：隣のリスト
>
> 辞書の値にリストをためると、つながりを表せる。キーが点、値が隣の点のリスト。線を1本読むたびに、両端のリストに相手を足す。


### お手本：世界のネットワーク

- 役割：出発と到着の列から、グラフを作る
- 入力：表`routes`
- 出力：グラフ`G`


In [ ]:
G = nx.from_pandas_edgelist(routes, '出発', '到着')
print(G.number_of_nodes(), G.number_of_edges())
print(nx.number_connected_components(G))


#### 1行ずつ読む

- 1行目：出発と到着の列から、グラフ`G`を作る
- 2行目：点（空港）と線の数。同じ2空港の路線は、航空会社が違っても1本にまとまる
- 3行目：互いに行き来できない、ばらばらの塊が8つある

`networkx`も、中では上のお手本と同じく、空港ごとの隣の一覧を辞書で持っています。`G['NGS']`で、長崎の隣が引けます。線の向き（出発と到着）は区別せず、どちらへも飛べる路線として扱います。


### お手本：いちばん大きい塊


In [ ]:
largest = max(nx.connected_components(G), key=len)
H = G.subgraph(largest).copy()
print(H.number_of_nodes(), H.number_of_edges())
print(sorted(H.neighbors('NGS')))


#### 1行ずつ読む

- 1行目：8つの塊のうち、空港の数（`len`）がいちばん大きい塊を取る
- 2行目：その空港だけのグラフ`H`。ほかの7つは、小さな島の路線など
- 3・4行目：点と線の数と、長崎の隣（直行便のある空港）


### 結果を見る前に、予想を書く

世界でいちばん多くの空港とつながっているのは、どこの空港でしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：空港を替える（予想 → 実行 → 比べる）

下のセルは、1つの空港の次数（つながっている空港の数）と、隣の空港の先頭10個を表示します。

まず、予想を書いてください。羽田は、いくつの空港とつながっているでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'NGS'`を`'HND'`に替えて実行してください（2か所とも）。


In [ ]:
# TODO: 'NGS' を 'HND' に替えて実行する

print(H.degree('HND'))
print(sorted(H.neighbors('HND'))[:10])


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。いくつでしたか。


（ここに答えを書く）


羽田は74空港とつながります。長崎（11）の7倍ほどです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表からつながりへ（1節） | 3,397空港・19,231本のネットワーク。長崎の隣は11空港 |
| 要の空港（2節） | ？（このあとの段で） |
| 世界の狭さ（3節） | ？（このあとの段で） |
| まとまり（4節） | ？（このあとの段で） |
| 切り方を替える（5節） | ？（このあとの段で） |

まだ分からないのは、世界の空港の中で、どこが要か、です。


---


## 2. どこが要か？


ここから、段2「要の空港」（2節）です。問いは、世界の空港のうち、どこが要か、です。

路線の一覧から、3,397空港・19,231本のネットワークを作りました。長崎の隣は11空港、羽田は74空港でした。空港ごとに、隣の数を比べます。

1つの点につながっている線の数を、次数といいます。空港なら、直行便のある空港の数です。次数が飛び抜けて大きい点を、ハブといいます。


### お手本：次数を数える


In [ ]:
deg = pd.Series(dict(H.degree()))
top8 = deg.sort_values(ascending=False)[:8]
print(top8)


#### 1行ずつ読む

- 1行目：空港ごとの次数を辞書にし（`dict`）、空港の記号を名前にした列にする
- 2行目：大きい順に並べ、上位8つを取る
- 3行目：アムステルダム（AMS）・フランクフルト（FRA）・パリ（CDG）・イスタンブール（IST）…


#### 中身を見る：都市の名前


In [ ]:
info.loc[top8.index, ['都市', '国']]


ヨーロッパのハブが4つと、アトランタ・北京・シカゴです。


In [ ]:
# @title 図：地図でハブを見る（中身は読み飛ばしてよい）
pos = info.reindex(list(H.nodes()))
plt.figure(figsize=(12, 6))
plt.scatter(pos['経度'], pos['緯度'], s=2, color='lightgray')
t = info.loc[top8.index]
plt.scatter(t['経度'], t['緯度'], s=top8.values * 1.2, color='tab:red')
for c in top8.index:
    plt.text(info.loc[c, '経度'] + 3, info.loc[c, '緯度'] + 3, c)
plt.xlabel('経度')
plt.ylabel('緯度')
plt.show()


### お手本：真ん中の空港


In [ ]:
print(deg.median())
print((deg <= 3).mean().round(3), (deg >= 100).sum())


#### 1行ずつ読む

- 1行目：次数の中央値
- 2行目：次数3以下の空港の割合と、100以上の空港の数

えっ、アムステルダムは248路線ですが、真ん中の空港は3路線です。半分以上（55.7%）の空港は、3空港以下としかつながっていません。100空港以上とつながるのは、64空港だけです。


### お手本：次数の分布


In [ ]:
plt.hist(deg, bins=50)
plt.yscale('log')
plt.xlabel('つながっている空港の数（次数）')
plt.ylabel('空港の数')
plt.show()


#### 1行ずつ読む

- 1行目：次数のヒストグラム（50本の棒）
- 2行目：縦軸を対数にする（1・10・100・1000が同じ間隔）。少ない棒も見えるようにする
- 3〜5行目：軸の名前を付けて表示する

次数の小さい空港がとても多く、大きい空港はほんの少しです。平均や真ん中だけでは、ネットワークの形は分かりません。少数のハブが、たくさんの空港を結びます。このような形を、スケールフリーと呼ぶことがあります。


> 重要：航空網の次数は、ほとんどの空港が小さく、少数のハブだけが飛び抜けて大きい。こうした分布では、平均と中央値が大きく離れる。


### TODO②：中央値を平均に（予想 → 実行 → 比べる）

下のセルは、次数の中央値を表示します。

まず、予想を書いてください。次数の平均は、中央値（3）と比べてどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`median`を`mean`に替えて実行してください。


In [ ]:
# TODO: median を mean に替えて実行する

print(deg.mean().round(1))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。平均はいくつでしたか。


（ここに答えを書く）


平均は11.3です。中央値3の4倍ほどです。少数のハブが、平均を大きく引き上げます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表からつながりへ（1節） | 3,397空港・19,231本のネットワーク。長崎の隣は11空港 |
| 要の空港（2節） | ハブはアムステルダム248路線。でも空港の中央値は3路線 |
| 世界の狭さ（3節） | ？（このあとの段で） |
| まとまり（4節） | ？（このあとの段で） |
| 切り方を替える（5節） | ？（このあとの段で） |

まだ分からないのは、小さな空港から、世界のどこまで何回で行けるか、です。


---


## 3. 何回で行ける？


ここから、段3「世界の狭さ」（3節）です。問いは、世界の空港どうしは、何回の乗り継ぎで行けるか、です。

次数の中央値は3で、ほとんどの空港は少しの空港としかつながりません。一方で、200以上の空港とつながるハブがあります。

2つの点を結ぶ道のうち、線の数がいちばん少ない道を、最短経路といいます。空港なら、乗り継ぎの回数がいちばん少ない行き方です。線の数を、その2点の距離とします（飛ぶ距離ではありません）。


### お手本：長崎からニューヨーク


In [ ]:
print(nx.shortest_path(H, 'NGS', 'JFK'))
print(nx.shortest_path_length(H, 'NGS', 'JFK'))


1行目は、長崎からニューヨーク（JFK）への最短経路です。2行目は、その線の数です。関西を経由して、2回で着きます。


### お手本：長崎から全部の空港へ

- 役割：長崎から、ほかのすべての空港までの最短の回数を数える
- 入力：グラフ`H`
- 出力：列`hops`


In [ ]:
hops = pd.Series(nx.single_source_shortest_path_length(H, 'NGS'))
print(hops.value_counts().sort_index().tolist())
print(hops.mean().round(2), hops.max())


#### 1行ずつ読む

- 1行目：長崎から、ほかのすべての空港までの最短の回数（辞書）を、列にする
- 2行目：回数ごとの空港の数（0回は長崎自身）
- 3行目：平均と、いちばん多い回数

3回で1,496空港、4回で1,172空港です。ほとんどの空港へ4回以内で行けます。


In [ ]:
hops.value_counts().sort_index().plot.bar()
plt.xlabel('乗り継ぎの回数（最短）')
plt.ylabel('空港の数')
plt.show()


### お手本：いちばん遠い空港


In [ ]:
far = hops[hops == hops.max()].index[0]
print(info.loc[far, ['都市', '国']].tolist())
print(nx.shortest_path(H, 'NGS', far))


長崎から最も遠いのは、カナダ北部のサリュイです。関西・パリ・モントリオールを経て、北の村々を伝って8回です。


### お手本：世界全体の平均

すべての空港の組で、最短の回数の平均を出します。計算に少し時間がかかります（十数秒）。


In [ ]:
print(round(nx.average_shortest_path_length(H), 2))


どの2つの空港も、平均4.1回の乗り継ぎで行けます。次数の小さい空港が多いのに、ハブを経由すると、すぐに世界へ出られます。点の数のわりに距離が短いネットワークを、スモールワールドといいます。


> 重要：ほとんどの空港は小さくても、少数のハブにつながるだけで、世界の空港のどこへでも数回で届く。


### TODO③：出発を替える（予想 → 実行 → 比べる）

下のセルは、1つの空港から、ほかのすべての空港までの最短の回数を数え、回数ごとの空港の数と平均を表示します。

まず、予想を書いてください。アムステルダム（ハブ）から出ると、平均の回数はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'NGS'`を`'AMS'`に替えて実行してください。


In [ ]:
# TODO: 'NGS' を 'AMS' に替えて実行する

h = pd.Series(nx.single_source_shortest_path_length(H, 'AMS'))
print(h.value_counts().sort_index().tolist())
print(h.mean().round(2))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。平均の回数は、どうなりましたか。


（ここに答えを書く）


アムステルダムからは平均2.54回です。1回で248空港、2回で1,613空港に届きます。ハブから出ると、1回分ほど近くなります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表からつながりへ（1節） | 3,397空港・19,231本のネットワーク。長崎の隣は11空港 |
| 要の空港（2節） | ハブはアムステルダム248路線。でも空港の中央値は3路線 |
| 世界の狭さ（3節） | どの2つの空港も、平均4.1回の乗り継ぎで行ける |
| まとまり（4節） | ？（このあとの段で） |
| 切り方を替える（5節） | ？（このあとの段で） |

まだ分からないのは、つながりだけで、地域のまとまりが見えるか、です。


---


## 4. 地域が分かれるか？


ここから、段4「まとまり」（4節）です。問いは、緯度・経度を使わず、路線のつながりだけで、地域のまとまりが見えるか、です。

少数のハブのおかげで、世界は平均4.1回で届きます。それでも、近い地域の空港どうしは、たくさんの路線で結ばれていそうです。

群の中の線が多く、群の間の線が少ない点の集まりを、コミュニティ（まとまり）といいます。第8・9回のクラスタリング（clustering）の、ネットワーク版です。今日は、つながりの濃さを測る物差しで群を作る道具（Louvain法）を使います。


### お手本：まとまりを探す

- 役割：Louvain法で群を作り、空港の数の多い順に並べる
- 入力：グラフ`H`
- 出力：群のリスト`com`（1つの群は空港の集まり）


In [ ]:
com = nx.community.louvain_communities(H, seed=0)
com = sorted(com, key=len, reverse=True)
sizes = []
for c in com[:8]:
    sizes.append(len(c))
print(len(com), sizes)


#### 1行ずつ読む

- 1行目：Louvain法で群を作る。群ごとの空港の集まりのリスト（乱数のシードは0）
- 2行目：空港の数の多い順に並べる（`key=len`は、空港の数で比べる指定）
- 3〜6行目：上位8群の空港の数をリストにため、群の数と並べて表示する


### お手本：群と地域を突き合わせる


In [ ]:
for j in range(4):
    reg = info.reindex(list(com[j]))['地域']
    print(j, len(com[j]), reg.value_counts().head(3).to_dict())


#### 1行ずつ読む

- 1行目：大きい4群を、1つずつ取る
- 2行目：その群の空港の地域を引く（`reindex`は、情報の無い空港も欠測で残す）
- 3行目：群の番号・空港の数・多い地域の上位3つ

群0はアメリカ大陸、群1はアジアとオーストラリア、群2はヨーロッパとアフリカの一部です。緯度・経度は使っていません。路線のつながりだけで、地域が出ました。長崎は群1（アジアとオーストラリア）に入ります。


In [ ]:
print('NGS' in com[1])


### お手本：地図に塗る


In [ ]:
group = {}
for j in range(len(com)):
    for x in com[j]:
        group[x] = j
pos = info.reindex(list(H.nodes()))
colors = []
for x in pos.index:
    colors.append(group[x])
plt.scatter(pos['経度'], pos['緯度'], s=3, c=colors)
plt.show()


#### 1行ずつ読む

- 1〜4行目：空港をキー、群の番号を値にした辞書を作る（入れ子の`for`、第9回）
- 5〜9行目：空港の経度・緯度を引き、群の番号を色のリストにためて、点を打つ
- 10行目：図を表示する


> 今日の重要概念（統計）：まとまり
>
> まとまり（コミュニティ）は、群の中の線が多く、群の間の線が少ない点の集まり。どんな物差しで「まとまり」を測るかで、見つかる群が変わる。


### TODO④：群の細かさを替える（予想 → 実行 → 比べる）

下のセルは、Louvain法で群を作り、群の数と上位8群の空港の数を表示します。`resolution`は群の細かさの設定（既定は1）です。

まず、予想を書いてください。0.5にすると、群の数はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`resolution=1`の`1`を`0.5`に替えて実行してください。


In [ ]:
# TODO: 1 を 0.5 に替えて実行する

com_x = nx.community.louvain_communities(H, seed=0, resolution=0.5)
com_x = sorted(com_x, key=len, reverse=True)
sizes_x = []
for c in com_x[:8]:
    sizes_x.append(len(c))
print(len(com_x), sizes_x)


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。群の数は、どうなりましたか。


（ここに答えを書く）


0.5にすると19群になり、大きな群がさらに大きくなります（866・782…）。群の数は、データだけでは決まりません。細かさの設定しだいです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表からつながりへ（1節） | 3,397空港・19,231本のネットワーク。長崎の隣は11空港 |
| 要の空港（2節） | ハブはアムステルダム248路線。でも空港の中央値は3路線 |
| 世界の狭さ（3節） | どの2つの空港も、平均4.1回の乗り継ぎで行ける |
| まとまり（4節） | つながりだけで群を作ると、大陸がほぼ出る（23群） |
| 切り方を替える（5節） | ？（このあとの段で） |

まだ分からないのは、別の物差しで切ると、群はどう変わるか、です。


---


## 5. 物差しを替えると？


ここから、段5「切り方を替える」（5節）です。問いは、別の物差しで群を切ると、何が見えるか。ハブを外すと、つながりはどうなるか、です。

つながりの濃さで群を作ると、大陸がほぼ出ました（23群）。群の数は、細かさの設定しだいで変わりました。多変量解析で学ぶ、スペクトルクラスタリングでも切ってみます。

スペクトルクラスタリングは、つながりの表（隣接行列）を固有値分解（eigendecomposition）し、少ない数の得点にしてから、k-meansで分けます。隣接行列は空港×空港の表で、つながっていれば1、いなければ0です。第11回の固有ベクトル（eigenvector）、第8回のk-meansの組み合わせです。


### お手本：隣接行列


In [ ]:
nodes = list(H.nodes())
A = nx.to_numpy_array(H, nodelist=nodes)
print(A.shape, int(A.sum()))
print(A[:5, :5])


3,397×3,397の表です。1の数は38,461で、線の数のほぼ2倍です。線は両方向に数えるからです（1本だけ、同じ空港に戻る路線があります）。


### お手本：スペクトルで切る

- 役割：隣接行列を、スペクトルクラスタリングで8群に分ける
- 入力：隣接行列`A`
- 出力：空港ごとの群の番号`lab`


In [ ]:
from sklearn.cluster import SpectralClustering
sc = SpectralClustering(8, affinity='precomputed', random_state=0,
                        assign_labels='discretize')
lab = sc.fit_predict(A)
print(sorted(np.bincount(lab).tolist(), reverse=True))


#### 1行ずつ読む

- 1〜3行目：8群に分ける準備。`affinity='precomputed'`で、渡す表をつながりの表として使う
- 4行目：隣接行列を渡して、空港ごとの群の番号を出す
- 5行目：群ごとの空港の数（`bincount`）を、大きい順に


In [ ]:
pos = info.reindex(nodes)
plt.figure(figsize=(12, 6))
plt.scatter(pos['経度'], pos['緯度'], s=3, c=lab, cmap='tab10')
plt.show()


えっ、3,072空港が1つの群です。残りの7群は、アラスカ・カナダ北部などの、小さな周辺の群です。

アラスカの小さな空港どうしは、互いに路線で結ばれ、外へは数本しかありません。この物差しは、群の間の線が少ない所で切ろうとするので、周辺の塊が切り出されます。大陸どうしは、ハブのおかげで線が多く、切りにくいのです。

Louvain法は群の中の線が偶然より多いかを測り、大陸が出ました。スペクトルクラスタリングは群の間の線の少なさを測り、周辺が出ました。どちらが正しいかではなく、何を「まとまり」と見るかが違います。


### お手本：ハブを外す

ハブは、世界を狭くしていました（3節）。では、ハブの空港が止まったら、つながりはどうなるでしょうか。次数の大きい順に20空港を外します。


In [ ]:
top = deg.sort_values(ascending=False).index
Hc = H.copy()
Hc.remove_nodes_from(top[:20])
print(nx.number_connected_components(Hc))
print(len(max(nx.connected_components(Hc), key=len)))


#### 1行ずつ読む

- 1・2行目：次数の大きい順の空港の記号と、ネットワークの写し（`H`は壊さない）
- 3行目：上位20空港を外す
- 4・5行目：ばらばらになった塊の数と、いちばん大きい塊の空港の数


比べるために、でたらめに20空港を外します。


In [ ]:
import random
random.seed(0)
Hr = H.copy()
Hr.remove_nodes_from(random.sample(nodes, 20))
print(nx.number_connected_components(Hr))


ハブを20外すと88個に分かれますが、でたらめに20外しても5個で、ほとんど崩れません。


> 重要：ハブは世界を狭くするが、ハブが止まるとネットワークはばらばらになる。でたらめな故障には強く、ハブを狙った止まり方には弱い。


### TODO⑤：外すハブの数を替える（予想 → 実行 → 比べる）

下のセルは、次数の大きい順に空港を外し、ばらばらになった塊の数と、いちばん大きい塊の空港の数を表示します。

まず、予想を書いてください。ハブを50外すと、いくつに分かれるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`20`を`50`に替えて実行してください。


In [ ]:
# TODO: 20 を 50 に替えて実行する

Hx = H.copy()
Hx.remove_nodes_from(top[:50])
print(nx.number_connected_components(Hx))
print(len(max(nx.connected_components(Hx), key=len)))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。いくつに分かれましたか。


（ここに答えを書く）


ハブを50外すと173個に分かれ、いちばん大きい塊も3,159空港に減ります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表からつながりへ（1節） | 3,397空港・19,231本のネットワーク。長崎の隣は11空港 |
| 要の空港（2節） | ハブはアムステルダム248路線。でも空港の中央値は3路線 |
| 世界の狭さ（3節） | どの2つの空港も、平均4.1回の乗り継ぎで行ける |
| まとまり（4節） | つながりだけで群を作ると、大陸がほぼ出る（23群） |
| 切り方を替える（5節） | スペクトルクラスタリングでは周辺の小さな群が切り出される。ハブを20外すと88個に |

5つの段がそろい、今日の問いに答えられます。


### 答え

要の空港はヨーロッパなどの少数のハブです。空港の真ん中は3路線で、世界は平均4.1回で届きます。路線のつながりだけで群を作ると、大陸がほぼ出ます。ただし、物差しを替えると、周辺の小さな群が切り出されます。何をまとまりと見るかで、群が変わります。


まだ言えないこと

- 2014年の路線。今の路線とは違う
- 便の数や飛ぶ距離、乗客の数は見ていない。路線があるかどうかだけ
- 路線の向き（片道だけの路線）も区別していない


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- データ：OpenFlightsの2014年の路線67,663行から、3,397空港・19,231本のネットワークを作った
- ハブ：アムステルダム248路線が最大。次数の中央値は3、平均は11.3
- 狭さ：どの2空港も平均4.1回の乗り継ぎ。長崎からの最遠はカナダ北部のサリュイ（8回）
- まとまり：Louvain法で23群。緯度・経度を使わずに、大陸がほぼ出た
- 驚き：スペクトルクラスタリングでは、周辺の小さな群が切り出された。物差しで群が変わる
- 限界：2014年の路線で、便の数や距離は見ていない


---


## 目標に答えられたか


- 今日の目標は「世界の空港のつながりから、要の空港と、まとまり（地域）が見えるか」でした。次数・最短経路・群の地図で答えられますか？
- 1節で、隣の一覧を作るとき、線1本につき、どこに何を足しましたか？
- 2節で、平均と中央値が大きく離れたのはなぜですか？
- 3節で、次数の小さい空港が多いのに、世界が狭いのはなぜですか？
- 4節で、緯度・経度を使わないのに、大陸が出たのはなぜですか？
- 5節で、Louvain法とスペクトルクラスタリングで群が違ったのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：ネットワークは、点と線でつながりを表したもの。次数でハブを、最短経路で世界の狭さを読み、まとまりを探す。何を「まとまり」と見るかで、見つかる群が変わる
- 重要概念（統計）：まとまり。群の中の線が多く、群の間の線が少ない点の集まり。どんな物差しで測るかで、見つかる群が変わる
- 重要概念（Python）：隣のリスト。辞書の値にリストをためると、つながりを表せる。キーが点、値が隣の点のリスト
- 次数の分布は、ほとんどが小さく少数だけ巨大。平均と中央値が離れる
- ハブが世界を狭くし、ハブが止まるとばらばらになる


---


## 次回へ


今日まで、データを読み、解析の手順を組み立て、結果を読んできました。では、自分で選んだデータで、何を明らかにしたいでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：まとまり。群の中の線が多く、群の間の線が少ない点の集まり。どんな物差しで測るかで、見つかる群が変わる（4・5節）
- Python：隣のリスト。辞書の値にリストをためると、つながりを表せる。キーが点、値が隣の点のリスト（1節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

まとまりなら、たとえば「学校の中の友だちの輪」を描けます。
クラスの中ではよく話し、クラスの外とは少し話します。「よく話す人の輪」で分けるとクラスが出ますが、「話す相手の少ない人」で分けると、ひとりでいる人の小さな輪が出ます。何で分けるかで、見える輪が変わります。4節と5節の2つの地図が手本になります。

</details>


<details><summary>詰まったら</summary>

隣のリストなら、たとえば「クラスの連絡網」を描けます。
名前ごとに、電話をかける相手の一覧を持ちます。新しくつながったら、2人の一覧の両方に、相手の名前を足します。名前を引けば、すぐに隣がわかります。1節の、5本の線から隣の一覧を作る例が手本になります。

</details>


---


## 発展（任意）


### 日本の空港だけ

世界のネットワークから、日本の空港だけを取り出します。国内線のつながりで、要の空港はどこでしょうか。


In [ ]:
jp = []
for x in info[info['国'] == 'Japan'].index:
    if x in H:
        jp.append(x)
J = H.subgraph(jp)
print(J.number_of_nodes(), J.number_of_edges())
print(pd.Series(dict(J.degree())).sort_values(ascending=False)[:5])


#### 1行ずつ読む

- 1〜4行目：国が日本の空港のうち、ネットワーク`H`にある空港を、リスト`jp`にためる
- 5行目：その空港だけのグラフ`J`（日本の空港どうしの路線だけが残る）
- 6・7行目：点と線の数と、次数の上位5空港

日本の62空港・175本です。羽田は49空港と国内線で結ばれ、新千歳・那覇・伊丹・中部が続きます。

試すなら、`'Japan'`を`'Indonesia'`（島の多い国）や`'Germany'`に替えて、要の空港を比べてみましょう。
